In [ ]:
# NOTEBOOK NAME
# PPIvarianceViewer.ipynb
# NOTEBOOK NAME

# OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# for projecting radar coordinates to lat and lon
from pyproj import Geod

# mapping things
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

# for adding lat/lon gridlines on plots
import matplotlib.ticker as mticker
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

# for adding a colourful topo base map to the CAPI plots
from custom_elevation import fetch_srtm, fetch_gebco_local
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# PPI WITH VARIANCES VERSION
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)


# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# # MASKING PARAMETERS
# RhoHVvarMax = 0.004    # Maximum RhoHV variance threshold
# RhoHVcountMin = 4      # Minimum RhoHV count threshold

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '12:00'
LoopEndTime   = '12:00'

# CHOOSE YOUR ELEVATION ANGLE
# ElevationAngle = 23  # [degrees] options are    0.5   0.8   1.4   2.4
                       #                         3.5   4.7   6.0   7.8
                       #                        10    13    17    23   and   32

# calculate the ground range when beam altitude reaches 20 km for a max ground radius of consideration 
RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))

# choose how far you want the extremes of the box in the plot from the radar
BoxRange = np.minimum(150.0, RangeOf20km ) # [km] set that maximum ground range as a limit to the plot, or 150 km as a max value


# elevation angle choice follow-on
# make sure that the chosen elevation angle is valide
AllElevationAngles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0])
assert ElevationAngle in AllElevationAngles, 'Please choose one of the available elevation angles: ' + \
                                            '0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0'

# CHOOSE YOUR VARIABLE
# Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [RhoHV]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# ra
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 10

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -5.0
    VarColourBar_max = 5.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0

elif (Var == 'RhoHV'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 0.05
    
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 10.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0
    # unofficial fill value: -0.00030518509475996325
    # unofficial fill value: 7.50038148136845

elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max =  30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'Z_variance'):
    VarName     = 'Reflectivity Variance'
    VarNameLong = 'corrected_reflectivity_3x3grid_variance'
    VarMinVal =  0 # [dBZ²]
    VarMaxVal = 65 # [dBZ²]
    VarUnit   = 'dBZ²'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarTickSpacing = 10.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'Z_count'):
    VarName     = 'Reflectivity Counts'
    VarNameLong = 'corrected_reflectivity_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -3.0
    VarColourBar_max = 10.0
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

elif (Var == 'V_variance'):
    VarName     = 'Velocity Variance'
    VarNameLong = 'corrected_velocity_3x3grid_variance'
    VarMinVal =  0 # [m²/s²]
    VarMaxVal = 1 # [m²/s²]
    VarUnit   = 'm²/s²'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 1.0
    VarTickSpacing = 0.1
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'V_count'):
    VarName     = 'Velocity Counts'
    VarNameLong = 'corrected_velocity_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 9
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    

elif (Var == 'RhoHV_variance'):
    VarName     = 'Correlation Coefficient Variance'
    VarNameLong = 'corrected_cross_correlation_ratio_3x3grid_variance'
    VarMinVal =  0 # 
    VarMaxVal = 0.01  # 
    VarUnit   = 'variance'
    VarFillValue = -32.0
    VarColourBar = 'viridis'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 0.01
    VarTickSpacing = 0.001
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'RhoHV_count'):
    VarName     = 'Correlation Coefficient Counts'
    VarNameLong = 'corrected_cross_correlation_ratio_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 9
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [RhoHV] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'HorzPPI'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi_with_variances.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # ALL IN ONE BLOCK

    # calculate the effective radius of the earth given standard refraction
    RadiusEarth = 6371000 # [m]
    RadiusEarthEff = RadiusEarth * (4/3)
    
    # load in a "geodesic calculator"
    geod = Geod(ellps='WGS84')
    
    # read in the PPI coordinates from the xarray dataframe
    Ranges  = RadarXR['range'].values
    AziDegs = RadarXR['azimuth'].values
    EleDegs = RadarXR['elevation'].values
    
    # read in the radar site location from the xarray dataframe
    SiteLat = float(RadarXR.latitude)
    SiteLon = float(RadarXR.longitude)
    SiteAlt = float(RadarXR.altitude)
    
    # convert angles to radians
    AziRads = np.deg2rad(AziDegs)[:,None]
    EleRads = np.deg2rad(EleDegs)[:,None]
    
    # I AM UNCERTAIN OF THIS TRIGONOMETRIC FORMULA
    
    # Distance from Centre of the Earth using 4/3 Earth-radius model
    DistsFromCOE = np.sqrt(    (Ranges**2)  +  (RadiusEarthEff**2)  +  (2.0 * Ranges * RadiusEarthEff * np.sin(EleRads))     )
    # Convert to altitude by subtracting out the earth and adding in the site elevation
    Alts = (DistsFromCOE - RadiusEarthEff) + SiteAlt
    
    # MAYBE THIS FORMULA SHOULD ALSO TAKE INTO ACCOUNT EARTH CURVATURE?
    
    # calculate range in distance along the surface of Earth
    GroundRanges = Ranges * np.cos(EleRads)  # distance along surface [m]
    # the last line adds a dimension of 1 at the end for some reason that I remove here:
    GroundRanges = np.squeeze(GroundRanges)
    
    # retrieve number of ranges and gates in the ppi file
    NumRays, NumGates = GroundRanges.shape
    
    # for i in range(NumRays):
        
    # Broadcast (repeat) radar site lon/lat and azimuth to match GroundRanges shape
    SiteLonRep = np.full_like(GroundRanges, SiteLon, dtype=float)
    SiteLatRep = np.full_like(GroundRanges, SiteLat, dtype=float)
    
    AziDegsRep= np.repeat(AziDegs[:, None], NumGates, axis=1)
    # AziDegsRep = np.full_like(GroundRanges, AziDegs, dtype=float)
    
    Lons, Lats, _ = geod.fwd(SiteLonRep, SiteLatRep, AziDegsRep, GroundRanges)
    
    
    # turn the calculated latitudes and longitudes and altitudes for each range gate into xarray elements 
    
    LatsData = xr.DataArray(Lats, dims=('time', 'range'), name='gate_latitude',
        attrs={
            'long_name': 'latitude of radar gates',
            'units': 'degrees_north'})
    
    LonsData = xr.DataArray(Lons, dims=('time', 'range'), name='gate_longitude',
        attrs={
            'long_name': 'longitude of radar gates',
            'units': 'degrees_east'})
    
    AltsData = xr.DataArray(Alts, dims=('time', 'range'), name='gate_altitude',
        attrs={
            'long_name': 'altitude of radar gates',
            'units': 'm',
            'standard_name': 'altitude'})
    
    
    # and add them as variables to your radar data
    RadarXR['gate_latitude']  = LatsData
    RadarXR['gate_longitude'] = LonsData
    RadarXR['gate_altitude']  = AltsData

    fig, ax = plt.subplots(figsize=(10, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})

    # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    
    # TERRAIN SHADING USING LOCAL GEBCO DEM
    lon_min, lon_max = float(RadarXR.gate_longitude.min()), float(RadarXR.gate_longitude.max())
    lat_min, lat_max = float(RadarXR.gate_latitude.min()), float(RadarXR.gate_latitude.max())

    try:
        # Path to your GEBCO NetCDF
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

        # Use the helper to get a subset over the radar domain
        dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is not None:
            # dem_da is an xarray.DataArray with coords lon, lat
            dem_lon = dem_da.lon.values
            dem_lat = dem_da.lat.values
            dem_data = dem_da.values

            # Make 2D lon/lat grids if necessary
            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            # Ensure we have some valid data
            valid = np.isfinite(dem_data)
            if not np.any(valid):
                raise ValueError('DEM has no finite values in this domain')

            colours = [
                '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
            
                '#c4dec2',  # 1: 0–200 m, pale green
                '#e4edc9',  # 2: 200–400 m, greenish-yellow
                '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
                '#e9d7bd',  # 4: 600–800 m, light tan
                '#ddc4aa',  # 5: 800–1000 m, tan
                '#cfb194',  # 6: 1000–1200 m, light brown
                '#b58f6e',  # 7: > 1200 m, darker brown
            ]
            
            bounds = [
                -1000.0,  # ocean below 0
                0.0,      # 0–200
                200.0,    # 200–400
                400.0,    # 400–600
                600.0,    # 600–800
                800.0,    # 800–1000
                1000.0,   # 1000–1200
                1200.0,   # > 1200
                5000.0,
            ]
            
            cmap_elev = ListedColormap(colours)
            norm = BoundaryNorm(bounds, len(colours), clip=True)
            
            # Plot as semi‑transparent background
            elev_plot = ax.pcolormesh(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                cmap=cmap_elev,
                norm=norm,
                alpha=1.0,
                transform=ccrs.PlateCarree(),
                # zorder=2,
            )

            # Draw 0 m contour as an accurate coastline
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[0.0],
                colors='black',
                linewidths=0.5,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            # Draw 400 m contour
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[400.0],
                colors='black',
                linewidths=0.3,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

        else:
            raise ValueError('GEBCO DEM returned None')

    except Exception as e:
        print(f'Terrain shading failed: {e}')
        print('Falling back to simple land/ocean shading')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION

    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE, select the "time coordinates" for the given elevation angle
    EleAngleMask = (RadarXR.elevation == ElevationAngle) # make a mask to only plot the one elevation angle
    Lats = RadarXR.gate_latitude[EleAngleMask]
    Lons = RadarXR.gate_longitude[EleAngleMask]
    PlotVar = RadarXR[VarNameLong][EleAngleMask]
    
    # when this is set to more than '0 neighbours' it removes the V=0.015599 whatever from the data since those are consdiered invalid in the counter func.
    BoxCountMask = (RadarXR['corrected_velocity_3x3grid_count'][EleAngleMask] >= 0) # make a mask to only plot points with many neighbours
    
    # NEW: Add RhoHV quality masks
    RhoHVvarMask = (RadarXR['corrected_cross_correlation_ratio_3x3grid_variance'][EleAngleMask] < RhoHVvarMax)
    RhoHVcountMask = (RadarXR['corrected_cross_correlation_ratio_3x3grid_count'][EleAngleMask] >= RhoHVcountMin)
    
    PlotVar = PlotVar.where(PlotVar != VarFillValue, np.nan) # get rid of the fill values in the variable you are plotting
    PlotVar = PlotVar.where(BoxCountMask) # get rid of all the low-neighbour points
    PlotVar = PlotVar.where(RhoHVvarMask) # mask out low RhoHV variance
    PlotVar = PlotVar.where(RhoHVcountMask) # mask out low RhoHV count


    GridViewer = ax.pcolormesh( Lons, Lats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, VarTickSpacing))  # tick spacing

    
    # Add MINOR gridlines (tenth degrees) - thin
    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
    gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

    # Add MID LEVEL gridlines (half degrees) - standard width with labels
    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
    gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

    # Add MAJOR gridlines (full degrees) - thick with labels
    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    
    # Format labels
    gl_mid.xformatter = LONGITUDE_FORMATTER
    gl_mid.yformatter = LATITUDE_FORMATTER
    
    gl_major.xformatter = LONGITUDE_FORMATTER
    gl_major.yformatter = LATITUDE_FORMATTER
    
    # Remove labels from top and right
    gl_mid.top_labels = False
    gl_mid.right_labels = False
    gl_mid.bottom_labels = True
    gl_mid.left_labels = True
    
    gl_major.top_labels = False
    gl_major.right_labels = False
    gl_major.bottom_labels = True
    gl_major.left_labels = True

    # calculate how many degrees this range is (different for latitude vs longitude)
    RadiusEarth = 6371 # [km]
    kmPerDegLat = (RadiusEarth * 2 * np.pi) /360
    kmPerDegLon = kmPerDegLat * np.cos( np.deg2rad(RadarXR.latitude))
    
    BoxRangeDegLat = BoxRange / kmPerDegLat
    BoxRangeDegLon = BoxRange / kmPerDegLon

    plt.xlim([float(RadarXR.longitude)  - BoxRangeDegLon, float(RadarXR.longitude)  + BoxRangeDegLon])
    plt.ylim([float(RadarXR.latitude)   - BoxRangeDegLat,  float(RadarXR.latitude)  + BoxRangeDegLat])
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nfor the ' + str(ElevationAngle) + '° Elevation Angle\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                 PlotType + str(ElevationAngle) + 'deg.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('doing')
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()

In [ ]:
Var = 'RhoHV'
ElevationAngle = 0.8

# MASKING PARAMETERS
RhoHVvarMax = 1.004    # Maximum RhoHV variance threshold
RhoHVcountMin = 0   # Minimum RhoHV count threshold

In [ ]:
# 5 x 3 GRID OF EVERY ELEVATION ANGLE VERSION
# PPI WITH VARIANCES VERSION
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

Var = 'RhoHV_variance'
ElevationAngle = 0.8

# MASKING PARAMETERS
RhoHVvarMax = 1.004    # Maximum RhoHV variance threshold
RhoHVcountMin = 0   # Minimum RhoHV count threshold

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# # MASKING PARAMETERS
# RhoHVvarMax = 0.004    # Maximum RhoHV variance threshold
# RhoHVcountMin = 4      # Minimum RhoHV count threshold

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 12
RadarDay   = 5
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '12:00'
LoopEndTime   = '12:00'

# calculate the ground range when beam altitude reaches 20 km for a max ground radius of consideration 
RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))

# choose how far you want the extremes of the box in the plot from the radar
BoxRange = np.minimum(150.0, RangeOf20km ) # [km] set that maximum ground range as a limit to the plot, or 150 km as a max value


# CHOOSE YOUR VARIABLE
# Var = 'Z'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [RhoHV]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# ra
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -10 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarFillValue = -32.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 10

elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 #10.000152 # [dB]
    VarMaxVal =  5 #10.000153 # [dB]
    VarUnit   = 'dB'
    VarColourBar =  'RdBu' #'plasma'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -5 #10.000152
    VarColourBar_max =  5 #10.000153
    VarFillValue = -15.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0 #0.0000002
    # unofficial fill value: 10.00015259254738

elif (Var == 'RhoHV'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarFillValue = 0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 0.05
    
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0#-0.00030519#7.500381  # [deg/ km]
    VarMaxVal = 8#-0.00030518#7.500382 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'gist_ncar'#'plasma'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0#-0.00030519#7.500381
    VarColourBar_max = 8#-0.00030518#7.500382
    VarFillValue = -5.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 1.0#0.000000002#0.0000002
    # unofficial fill value: -0.00030518509475996325
    # unofficial fill value: 7.50038148136845

elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.0
    VarColourBar_max = 30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max =  30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    VarTickSpacing = 5.0

elif (Var == 'Z_variance'):
    VarName     = 'Reflectivity Variance'
    VarNameLong = 'corrected_reflectivity_3x3grid_variance'
    VarMinVal =  0 # [dBZ²]
    VarMaxVal = 65 # [dBZ²]
    VarUnit   = 'dBZ²'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarTickSpacing = 10.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'Z_count'):
    VarName     = 'Reflectivity Counts'
    VarNameLong = 'corrected_reflectivity_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -3.0
    VarColourBar_max = 10.0
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

elif (Var == 'V_variance'):
    VarName     = 'Velocity Variance'
    VarNameLong = 'corrected_velocity_3x3grid_variance'
    VarMinVal =  0 # [m²/s²]
    VarMaxVal = 1 # [m²/s²]
    VarUnit   = 'm²/s²'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 1.0
    VarTickSpacing = 0.1
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'V_count'):
    VarName     = 'Velocity Counts'
    VarNameLong = 'corrected_velocity_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 9
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
    

elif (Var == 'RhoHV_variance'):
    VarName     = 'Correlation Coefficient Variance'
    VarNameLong = 'corrected_cross_correlation_ratio_3x3grid_variance'
    VarMinVal =  0 # 
    VarMaxVal = 0.01  # 
    VarUnit   = 'correlation²'
    VarFillValue = -32.0
    VarColourBar = 'viridis'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 0.01
    VarTickSpacing = 0.001
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'RhoHV_count'):
    VarName     = 'Correlation Coefficient Counts'
    VarNameLong = 'corrected_cross_correlation_ratio_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 9
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

elif (Var == 'ZDR_variance'):
    VarName     = 'Corrected Differential Reflectivity Variance'
    VarNameLong = 'corrected_differential_reflectivity_3x3grid_variance'
    VarMinVal =  0 # [dB²]
    VarMaxVal = 5  # [dB²]
    VarUnit   = 'dB²'
    VarFillValue = -9999.0 # [dBZ²]
    VarColourBar = 'viridis'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 5
    VarTickSpacing = 1
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'ZDR_count'):
    VarName     = 'Corrected Differential Reflectivity Counts'
    VarNameLong = 'corrected_differential_reflectivity_3x3grid_count'
    VarMinVal =  0 # [cells]
    VarMaxVal = 9 # [cells]
    VarUnit   = 'cells'
    VarFillValue = -32.0
    VarColourBar = 'RdPu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 9
    VarTickSpacing = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)

else:
    raise ValueError("Input Variable '" + Var + "' not available")


# remember these plots are horizontal cross sections
PlotType = 'HorzPPI'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi_with_variances.nc' #
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # ALL IN ONE BLOCK

    # calculate the effective radius of the earth given standard refraction
    RadiusEarth = 6371000 # [m]
    RadiusEarthEff = RadiusEarth * (4/3)
    
    # load in a "geodesic calculator"
    geod = Geod(ellps='WGS84')
    
    # read in the PPI coordinates from the xarray dataframe
    Ranges  = RadarXR['range'].values
    AziDegs = RadarXR['azimuth'].values
    EleDegs = RadarXR['elevation'].values
    
    # read in the radar site location from the xarray dataframe
    SiteLat = float(RadarXR.latitude)
    SiteLon = float(RadarXR.longitude)
    SiteAlt = float(RadarXR.altitude)
    
    # convert angles to radians
    AziRads = np.deg2rad(AziDegs)[:,None]
    EleRads = np.deg2rad(EleDegs)[:,None]
    
    # I AM UNCERTAIN OF THIS TRIGONOMETRIC FORMULA
    
    # Distance from Centre of the Earth using 4/3 Earth-radius model
    DistsFromCOE = np.sqrt(    (Ranges**2)  +  (RadiusEarthEff**2)  +  (2.0 * Ranges * RadiusEarthEff * np.sin(EleRads))     )
    # Convert to altitude by subtracting out the earth and adding in the site elevation
    Alts = (DistsFromCOE - RadiusEarthEff) + SiteAlt
    
    # MAYBE THIS FORMULA SHOULD ALSO TAKE INTO ACCOUNT EARTH CURVATURE?
    
    # calculate range in distance along the surface of Earth
    GroundRanges = Ranges * np.cos(EleRads)  # distance along surface [m]
    # the last line adds a dimension of 1 at the end for some reason that I remove here:
    GroundRanges = np.squeeze(GroundRanges)
    
    # retrieve number of ranges and gates in the ppi file
    NumRays, NumGates = GroundRanges.shape
    
    # for i in range(NumRays):
        
    # Broadcast (repeat) radar site lon/lat and azimuth to match GroundRanges shape
    SiteLonRep = np.full_like(GroundRanges, SiteLon, dtype=float)
    SiteLatRep = np.full_like(GroundRanges, SiteLat, dtype=float)
    
    AziDegsRep= np.repeat(AziDegs[:, None], NumGates, axis=1)
    # AziDegsRep = np.full_like(GroundRanges, AziDegs, dtype=float)
    
    Lons, Lats, _ = geod.fwd(SiteLonRep, SiteLatRep, AziDegsRep, GroundRanges)
    
    
    # turn the calculated latitudes and longitudes and altitudes for each range gate into xarray elements 
    
    LatsData = xr.DataArray(Lats, dims=('time', 'range'), name='gate_latitude',
        attrs={
            'long_name': 'latitude of radar gates',
            'units': 'degrees_north'})
    
    LonsData = xr.DataArray(Lons, dims=('time', 'range'), name='gate_longitude',
        attrs={
            'long_name': 'longitude of radar gates',
            'units': 'degrees_east'})
    
    AltsData = xr.DataArray(Alts, dims=('time', 'range'), name='gate_altitude',
        attrs={
            'long_name': 'altitude of radar gates',
            'units': 'm',
            'standard_name': 'altitude'})
    
    
    # and add them as variables to your radar data
    RadarXR['gate_latitude']  = LatsData
    RadarXR['gate_longitude'] = LonsData
    RadarXR['gate_altitude']  = AltsData

        # --- ALL-ELEVATION SUBPLOT FIGURE ---
    AllElevationAngles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0])
    nCols = 3
    nRows = 5  # 3x5 = 15 panels; 13 angles + 2 empty

    fig, axes = plt.subplots(
        nRows, nCols,
        figsize=(nCols * 7, nRows * 6),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    axes_flat = axes.flatten()

    for idx, ElevationAngle in enumerate(AllElevationAngles):

        ax = axes_flat[idx]

        # --- Box range for this elevation angle ---
        RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))
        BoxRange = np.minimum(150.0, RangeOf20km)

        # --- Coarser GEBCO subset (every 3rd point) ---
        lon_min = float(RadarXR.longitude) - BoxRange / (6371 * np.cos(np.deg2rad(float(RadarXR.latitude))) * 2 * np.pi / 360)
        lon_max = float(RadarXR.longitude) + BoxRange / (6371 * np.cos(np.deg2rad(float(RadarXR.latitude))) * 2 * np.pi / 360)
        lat_min = float(RadarXR.latitude)  - BoxRange / (6371 * 2 * np.pi / 360)
        lat_max = float(RadarXR.latitude)  + BoxRange / (6371 * 2 * np.pi / 360)

        try:
            gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
            dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

            if dem_da is not None:
                dem_lon = dem_da.lon.values[::3]   # coarser: every 3rd point
                dem_lat = dem_da.lat.values[::3]
                dem_data = dem_da.values[::3, ::3]

                if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                    dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
                else:
                    dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

                valid = np.isfinite(dem_data)
                if not np.any(valid):
                    raise ValueError('DEM has no finite values in this domain')

                colours = [
                    '#dde4e8',
                    '#c4dec2', '#e4edc9', '#f3f0cf', '#e9d7bd',
                    '#ddc4aa', '#cfb194', '#b58f6e',
                ]
                bounds = [-1000.0, 0.0, 200.0, 400.0, 600.0, 800.0, 1000.0, 1200.0, 5000.0]
                cmap_elev = ListedColormap(colours)
                norm_elev = BoundaryNorm(bounds, len(colours), clip=True)

                ax.pcolormesh(
                    dem_lon_2d, dem_lat_2d, dem_data,
                    cmap=cmap_elev, norm=norm_elev,
                    alpha=1.0, transform=ccrs.PlateCarree()
                )
                ax.contour(
                    dem_lon_2d, dem_lat_2d, dem_data,
                    levels=[0.0], colors='black', linewidths=0.5,
                    transform=ccrs.PlateCarree(), zorder=15
                )
                ax.contour(
                    dem_lon_2d, dem_lat_2d, dem_data,
                    levels=[400.0], colors='black', linewidths=0.3,
                    transform=ccrs.PlateCarree(), zorder=15
                )
            else:
                raise ValueError('GEBCO DEM returned None')

        except Exception as e:
            ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
            ax.add_feature(cfeature.LAND,  facecolor='#E8E8E8',   alpha=0.3, zorder=2)

        ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)

        # --- Radar data for this elevation angle ---
        EleAngleMask = (np.abs(RadarXR.elevation - ElevationAngle) < 0.001)
        Lats_plot = RadarXR.gate_latitude[EleAngleMask]
        Lons_plot = RadarXR.gate_longitude[EleAngleMask]
        PlotVar   = RadarXR[VarNameLong][EleAngleMask]

        # BoxCountMask   = (RadarXR['corrected_velocity_3x3grid_count'][EleAngleMask] >= 0)
        # RhoHVvarMask   = (RadarXR['corrected_cross_correlation_ratio_3x3grid_variance'][EleAngleMask] < RhoHVvarMax)
        # RhoHVcountMask = (RadarXR['corrected_cross_correlation_ratio_3x3grid_count'][EleAngleMask] >= RhoHVcountMin)

        PlotVar = PlotVar.where(PlotVar != VarFillValue, np.nan)
        # PlotVar = PlotVar.where(BoxCountMask)
        # PlotVar = PlotVar.where(RhoHVvarMask)
        # PlotVar = PlotVar.where(RhoHVcountMask)

        # print(f'Occurences of KDP for {ElevationAngle} Angle: {occurences(PlotVar)}')
 
        GridViewer = ax.pcolormesh(
            Lons_plot, Lats_plot, PlotVar,
            cmap=VarColourBar, norm=VarColourBar_norm,
            shading='auto', transform=ccrs.PlateCarree()
        )

        cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']', fraction=0.044, shrink=0.99, pad=0.04)

        cbar.ax.set_ylim(VarMinVal, VarMaxVal)
        cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, VarTickSpacing))

        # --- Gridlines ---
        gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
        gl_minor.xlocator = mticker.MultipleLocator(0.1)
        gl_minor.ylocator = mticker.MultipleLocator(0.1)

        gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
        gl_major.xlocator = mticker.MultipleLocator(1.0)
        gl_major.ylocator = mticker.MultipleLocator(1.0)
        gl_major.xformatter = LONGITUDE_FORMATTER
        gl_major.yformatter = LATITUDE_FORMATTER
        gl_major.top_labels    = False
        gl_major.right_labels  = False

        # --- Axis limits ---
        RadiusEarth   = 6371
        kmPerDegLat   = (RadiusEarth * 2 * np.pi) / 360
        kmPerDegLon   = kmPerDegLat * np.cos(np.deg2rad(float(RadarXR.latitude)))
        BoxRangeDegLat = BoxRange / kmPerDegLat
        BoxRangeDegLon = BoxRange / kmPerDegLon

        ax.set_xlim([float(RadarXR.longitude) - BoxRangeDegLon, float(RadarXR.longitude) + BoxRangeDegLon])
        ax.set_ylim([float(RadarXR.latitude)  - BoxRangeDegLat, float(RadarXR.latitude)  + BoxRangeDegLat])

        ax.set_title(f'{ElevationAngle}° Elevation Angle', fontsize=24)

    # --- Hide unused subplots (panels 14 and 15) ---
    for idx in range(len(AllElevationAngles), nRows * nCols):
        axes_flat[idx].set_visible(False)

    # --- Overall figure title ---
    fig.suptitle(
        f'{VarName}\nFor the {RadarSiteName} Radar\nOn '
        f'{RadarFileDate[0:4]}-{RadarFileDate[4:6]}-{RadarFileDate[6:8]} at '
        f'{str(RadarFileTime)[0:2]}:{str(RadarFileTime)[2:4]}:{str(RadarFileTime)[4:6]} UTC',
        fontsize=32, y=0.9975
    )

    plt.tight_layout()

    # --- Save ---
    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
    SaveFile   = (RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' +
                  VarNameLong + '_' + PlotType + 'Everydeg.png')
    SavePath   = SaveFolder + SaveFile

    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)

    plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi=150)
    plt.close()


In [ ]:
VarNameLong

In [ ]:
SavePath

In [ ]:
# CHAD FUNCTION TO SHOW A RELATIONSHIP BETWEEN VARIABLE AND VALID CELLS COUNTED

VarNameLong = 'corrected_cross_correlation_ratio'

NeighbourCounts = RadarXR[VarNameLong + '_3x3grid_count'][EleAngleMask].values.ravel()
VariableToPlot  = RadarXR[VarNameLong+ '_3x3grid_variance'][EleAngleMask].values.ravel()

# Use the count NaN mask as the single source of truth for validity
valid_mask      = ~np.isnan(NeighbourCounts)
NeighbourCounts = NeighbourCounts[valid_mask]
VariableToPlot  = VariableToPlot[valid_mask]

# Bin edges: counts are 1-9 so we centre each integer bin
count_edges    = np.arange(0.5, 10.5, 1)          # 9 bins centred on 1,2,...,9
variable_edges = np.linspace(VarColourBar_min,  VarColourBar_max, (int( (VarColourBar_max - VarColourBar_min)/VarTickSpacing))*4+1 ) 
# 2 bins for each tick space

# Total valid points for normalisation
total_valid = np.sum(~np.isnan(NeighbourCounts))

h, xedges, yedges = np.histogram2d(
    NeighbourCounts, VariableToPlot,
    bins=[count_edges, variable_edges]
)

h = (h / total_valid) * 100  # convert to percent
h[h == 0] = np.nan


fig = plt.figure(figsize=(12, 8))
gs  = fig.add_gridspec(2, 2, height_ratios=[5, 1], width_ratios=[1, 6], hspace=0.05, wspace=0.05)

ax_pdf  = fig.add_subplot(gs[0, 0])
ax_main = fig.add_subplot(gs[0, 1], sharey=ax_pdf)
ax_hist = fig.add_subplot(gs[1, 1], sharex=ax_main)

# --- Left PDF ---
variable_bins  = np.linspace(VarColourBar_min,  VarColourBar_max, (int( (VarColourBar_max - VarColourBar_min)/VarTickSpacing))*4+1 )   # same bins as main plot
variable_cents = (variable_bins[:-1] + variable_bins[1:]) / 2
var_counts, _  = np.histogram(VariableToPlot, bins=variable_bins)
var_pct        = (var_counts / total_valid) * 100

bin_width = variable_bins[1] - variable_bins[0]
ax_pdf.barh(variable_cents, var_pct, height=bin_width, color=[0.0, 0.7, 0.7], edgecolor=None)

ax_pdf.set_ylabel(VarName + ' [' + VarUnit + ']')
ax_pdf.set_xlabel('% of valid points',fontsize='6')
ax_pdf.set_ylim(VarColourBar_min,  VarColourBar_max)
ax_pdf.set_facecolor('gray')
ax_pdf.invert_xaxis()  # so the PDF grows leftward, away from the heatmap

for v in np.arange(VarColourBar_min,  VarColourBar_max+VarTickSpacing, VarTickSpacing):
    ax_pdf.axhline(v, color='white', linewidth=0.4, alpha=0.6)
for v in np.arange(VarColourBar_min, VarColourBar_max+VarTickSpacing, VarTickSpacing*2):
    ax_pdf.axhline(v, color='white', linewidth=1.0, alpha=0.8)

pdf_max  = np.ceil(np.max(var_pct) / 10) * 10
ax_pdf.set_xlim(pdf_max, 0)  # inverted since axis is flipped

for hl in np.arange(0, 101, 5):
    ax_pdf.axvline(hl, color='white', linewidth=0.2, alpha=0.6)
for hl in np.arange(0, 101, 10):
    ax_pdf.axvline(hl, color='white', linewidth=0.5, alpha=0.8)


# --- Main heatmap ---
ax_main.pcolormesh(xedges, yedges, h.T, cmap=make_ChadMapZ())
cbar = plt.colorbar(ax_main.collections[0], ax=ax_main)
cbar.set_label('% of valid points')
ax_main.set_xticks(np.arange(1, 10))
ax_main.set_title(VarName + ' vs Neighbour Count for ' + RadarSiteName + ' Radar\nfor the ' + str(ElevationAngle) + '° Elevation Angle\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
ax_main.set_facecolor('gray')
plt.setp(ax_main.get_xticklabels(), visible=False)
plt.setp(ax_main.get_yticklabels(), visible=False)

for v in np.arange(VarColourBar_min,  VarColourBar_max+VarTickSpacing, VarTickSpacing):
    ax_main.axhline(v, color='white', linewidth=0.4, alpha=0.6)
for v in np.arange(VarColourBar_min, VarColourBar_max+VarTickSpacing, VarTickSpacing*2):
    ax_main.axhline(v, color='white', linewidth=1.0, alpha=0.8)

# --- Bottom histogram ---
count_values = np.arange(1, 10)
count_totals = [(np.sum(NeighbourCounts == c) / total_valid) * 100 for c in count_values]

ax_hist.bar(count_values, count_totals, width=0.8, color=[0.0, 0.7, 0.7], edgecolor='white')
ax_hist.set_xlabel('Neighbour Count')
ax_hist.set_ylabel('% of valid points',fontsize='6')
ax_hist.set_xticks(np.arange(1, 10))
ax_hist.set_facecolor('gray')

hist_max = np.ceil(np.max(count_totals) / 10) * 10
ax_hist.set_ylim(0, hist_max)

for v in np.arange(0, hist_max + 1, 5):
    ax_hist.axhline(v, color='white', linewidth=0.2, alpha=0.6)
for v in np.arange(0, hist_max + 1, 10):
    ax_hist.axhline(v, color='white', linewidth=0.5, alpha=0.8)

# --- Alignment: match all axes widths/positions to exclude colourbar ---
ax_main_pos = ax_main.get_position()
ax_hist.set_position([ax_main_pos.x0, ax_hist.get_position().y0,
                      ax_main_pos.width, ax_hist.get_position().height])
ax_pdf_pos  = ax_pdf.get_position()
ax_pdf.set_position([ax_pdf_pos.x0, ax_main_pos.y0,
                     ax_pdf_pos.width, ax_main_pos.height])

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/VarHists/' + RadarIDno + '/' + RadarFileDate + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
             PlotType + str(ElevationAngle) + 'deg.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
np.arange(VarColourBar_min,  VarColourBar_max+1, VarTickSpacing)

In [ ]:
# WITHOUT MASKS?

# PPI WITH VARIANCES VERSION
# HORIZONTAL CROSS SECTION PLOTTING (LOADS IN FROM NET CDF FILES STORED IN SCRATCH)

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 14
# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '12:00'
LoopEndTime   = '12:00'

# CHOOSE YOUR ELEVATION ANGLE
ElevationAngle = 0.5  # [degrees] options are    0.5   0.8   1.4   2.4
                       #                         3.5   4.7   6.0   7.8
                       #                        10    13    17    23   and   32

# calculate the ground range when beam altitude reaches 20 km for a max ground radius of consideration 
RangeOf20km = 20 / (np.tan(np.deg2rad(ElevationAngle)))

# choose how far you want the extremes of the box in the plot from the radar
BoxRange = np.minimum(150.0, RangeOf20km ) # [km] set that maximum ground range as a limit to the plot, or 150 km as a max value


# elevation angle choice follow-on
# make sure that the chosen elevation angle is valide
AllElevationAngles = np.array([0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0])
assert ElevationAngle in AllElevationAngles, 'Please choose one of the available elevation angles: ' + \
                                            '0.5, 0.8, 1.4, 2.4, 3.5, 4.7, 6.0, 7.8, 10.0, 13.0, 17.0, 23.0, 32.0'

# CHOOSE YOUR VARIABLE
Var = 'V'
# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [RhoHV]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
...
# [V]         'corrected_velocity'

# VARIABLES NOT YET INCLUDED IN THE CODE
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
...
# [AzSh]      'azshear'



# USER CHOICE FOLLOW-ON SECTION

# ra
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD


# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal =  -20 # [dBZ]
    VarMaxVal =   65 # [dBZ]
    VarUnit   = 'dBZ'
    VarFillValue = -32.0
    VarColourBar = make_ChadMapZ()
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max = 100.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarFillValue = -15
    VarColourBar = 'RdBu'
elif (Var == 'RhoHV'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarFillValue = 0
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0.8
    VarColourBar_max = 1.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarFillValue = -5
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 10
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarFillValue = 0
    VarColourBar = 'nipy_spectral'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = 0
    VarColourBar_max = 30
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarFillValue = -300
    VarColourBar = 'RdBu_r'
    # fix the colour bar to all values no matter which range you choose to view
    VarColourBar_min = -30.0
    VarColourBar_max =  30.0
    VarColourBar_norm = Normalize(vmin=VarColourBar_min, vmax=VarColourBar_max)
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [RhoHV] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

    

# remember these plots are horizontal cross sections
PlotType = 'HorzPPI'
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin

for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)
    # xgrid = xr.open_dataset('/scratch/v46/sg3241/tmp/NetCDFs/RadarGrids/' + RadarIDno + '/' + YYYY + '/' + MM + '/' + DD + '/' + \
    #                         RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.nc')


    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi_with_variances.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue

    # ALL IN ONE BLOCK

    # calculate the effective radius of the earth given standard refraction
    RadiusEarth = 6371000 # [m]
    RadiusEarthEff = RadiusEarth * (4/3)
    
    # load in a "geodesic calculator"
    geod = Geod(ellps='WGS84')
    
    # read in the PPI coordinates from the xarray dataframe
    Ranges  = RadarXR['range'].values
    AziDegs = RadarXR['azimuth'].values
    EleDegs = RadarXR['elevation'].values
    
    # read in the radar site location from the xarray dataframe
    SiteLat = float(RadarXR.latitude)
    SiteLon = float(RadarXR.longitude)
    SiteAlt = float(RadarXR.altitude)
    
    # convert angles to radians
    AziRads = np.deg2rad(AziDegs)[:,None]
    EleRads = np.deg2rad(EleDegs)[:,None]
    
    # I AM UNCERTAIN OF THIS TRIGONOMETRIC FORMULA
    
    # Distance from Centre of the Earth using 4/3 Earth-radius model
    DistsFromCOE = np.sqrt(    (Ranges**2)  +  (RadiusEarthEff**2)  +  (2.0 * Ranges * RadiusEarthEff * np.sin(EleRads))     )
    # Convert to altitude by subtracting out the earth and adding in the site elevation
    Alts = (DistsFromCOE - RadiusEarthEff) + SiteAlt
    
    # MAYBE THIS FORMULA SHOULD ALSO TAKE INTO ACCOUNT EARTH CURVATURE?
    
    # calculate range in distance along the surface of Earth
    GroundRanges = Ranges * np.cos(EleRads)  # distance along surface [m]
    # the last line adds a dimension of 1 at the end for some reason that I remove here:
    GroundRanges = np.squeeze(GroundRanges)
    
    # retrieve number of ranges and gates in the ppi file
    NumRays, NumGates = GroundRanges.shape
    
    # for i in range(NumRays):
        
    # Broadcast (repeat) radar site lon/lat and azimuth to match GroundRanges shape
    SiteLonRep = np.full_like(GroundRanges, SiteLon, dtype=float)
    SiteLatRep = np.full_like(GroundRanges, SiteLat, dtype=float)
    
    AziDegsRep= np.repeat(AziDegs[:, None], NumGates, axis=1)
    # AziDegsRep = np.full_like(GroundRanges, AziDegs, dtype=float)
    
    Lons, Lats, _ = geod.fwd(SiteLonRep, SiteLatRep, AziDegsRep, GroundRanges)
    
    
    # turn the calculated latitudes and longitudes and altitudes for each range gate into xarray elements 
    
    LatsData = xr.DataArray(Lats, dims=('time', 'range'), name='gate_latitude',
        attrs={
            'long_name': 'latitude of radar gates',
            'units': 'degrees_north'})
    
    LonsData = xr.DataArray(Lons, dims=('time', 'range'), name='gate_longitude',
        attrs={
            'long_name': 'longitude of radar gates',
            'units': 'degrees_east'})
    
    AltsData = xr.DataArray(Alts, dims=('time', 'range'), name='gate_altitude',
        attrs={
            'long_name': 'altitude of radar gates',
            'units': 'm',
            'standard_name': 'altitude'})
    
    
    # and add them as variables to your radar data
    RadarXR['gate_latitude']  = LatsData
    RadarXR['gate_longitude'] = LonsData
    RadarXR['gate_altitude']  = AltsData

    fig, ax = plt.subplots(figsize=(10, 8), 
                           subplot_kw={'projection': ccrs.PlateCarree()})

    # gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    # EXPERIMENTAL TOPO SHADING SECTION
    
    # TERRAIN SHADING USING LOCAL GEBCO DEM
    lon_min, lon_max = float(RadarXR.gate_longitude.min()), float(RadarXR.gate_longitude.max())
    lat_min, lat_max = float(RadarXR.gate_latitude.min()), float(RadarXR.gate_latitude.max())

    try:
        # Path to your GEBCO NetCDF
        gebco_path = '/home/563/sg3241/QueenslandElevationGEBCO.nc'

        # Use the helper to get a subset over the radar domain
        dem_da = fetch_gebco_local(gebco_path, lon_min, lon_max, lat_min, lat_max)

        if dem_da is not None:
            # dem_da is an xarray.DataArray with coords lon, lat
            dem_lon = dem_da.lon.values
            dem_lat = dem_da.lat.values
            dem_data = dem_da.values

            # Make 2D lon/lat grids if necessary
            if dem_lon.ndim == 1 and dem_lat.ndim == 1:
                dem_lon_2d, dem_lat_2d = np.meshgrid(dem_lon, dem_lat)
            else:
                dem_lon_2d, dem_lat_2d = dem_lon, dem_lat

            # Ensure we have some valid data
            valid = np.isfinite(dem_data)
            if not np.any(valid):
                raise ValueError('DEM has no finite values in this domain')

            colours = [
                '#dde4e8',  # 0: pale blue-grey (ocean, < 0 m)
            
                '#c4dec2',  # 1: 0–200 m, pale green
                '#e4edc9',  # 2: 200–400 m, greenish-yellow
                '#f3f0cf',  # 3: 400–600 m, pale yellow-beige
                '#e9d7bd',  # 4: 600–800 m, light tan
                '#ddc4aa',  # 5: 800–1000 m, tan
                '#cfb194',  # 6: 1000–1200 m, light brown
                '#b58f6e',  # 7: > 1200 m, darker brown
            ]
            
            bounds = [
                -1000.0,  # ocean below 0
                0.0,      # 0–200
                200.0,    # 200–400
                400.0,    # 400–600
                600.0,    # 600–800
                800.0,    # 800–1000
                1000.0,   # 1000–1200
                1200.0,   # > 1200
                5000.0,
            ]
            
            cmap_elev = ListedColormap(colours)
            norm = BoundaryNorm(bounds, len(colours), clip=True)
            
            # Plot as semi‑transparent background
            elev_plot = ax.pcolormesh(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                cmap=cmap_elev,
                norm=norm,
                alpha=1.0,
                transform=ccrs.PlateCarree(),
                # zorder=2,
            )

            # Draw 0 m contour as an accurate coastline
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[0.0],
                colors='black',
                linewidths=0.5,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            # Draw 400 m contour
            coast_contour = ax.contour(
                dem_lon_2d,
                dem_lat_2d,
                dem_data,
                levels=[400.0],
                colors='black',
                linewidths=0.3,
                transform=ccrs.PlateCarree(),
                zorder=15,  # above radar and topo
            )

            print('Terrain shading (GEBCO, discrete bands) + coastline loaded successfully')

        else:
            raise ValueError('GEBCO DEM returned None')

    except Exception as e:
        print(f'Terrain shading failed: {e}')
        print('Falling back to simple land/ocean shading')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3, zorder=1)
        ax.add_feature(cfeature.LAND, facecolor='#E8E8E8', alpha=0.3, zorder=2)

    ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.5, zorder=3)
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION
    # END EXPERIMENTAL TOPO SHADING SECTION

    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE
    # ACTUAL PLOTTING RIGHT HERE, select the "time coordinates" for the given elevation angle
    EleAngleMask = (RadarXR.elevation == ElevationAngle) # make a mask to only plot the one elevation angle
    Lats = RadarXR.gate_latitude[EleAngleMask]
    Lons = RadarXR.gate_longitude[EleAngleMask]
    PlotVar = RadarXR[VarNameLong][EleAngleMask]

    # BoxCountMask = (RadarXR[VarNameLong+'_3x3grid_count'][EleAngleMask] >= 4) # make a mask to only plot points with many neighbours

    PlotVar = PlotVar.where(PlotVar != VarFillValue, np.nan) # get rid of the fill values in the variable you are plotting
    # PlotVar = PlotVar.where(BoxCountMask) # get rid of all the low-neighbour points
    
    GridViewer = ax.pcolormesh( Lons, Lats, PlotVar, cmap=VarColourBar, norm=VarColourBar_norm, shading='auto', transform=ccrs.PlateCarree() )

    cbar = plt.colorbar(GridViewer, ax=ax, label=VarName + ' [' + VarUnit + ']')
    cbar.ax.set_ylim(VarMinVal, VarMaxVal)          # zoom the displayed range on the colorbar
    cbar.set_ticks(np.arange(VarMinVal, VarMaxVal, 10))  # tick every 10 dBZ

    
    # Add MINOR gridlines (tenth degrees) - thin
    gl_minor = ax.gridlines(draw_labels=False, alpha=0.8, zorder=11, linewidth=0.2)
    gl_minor.xlocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees
    gl_minor.ylocator = mticker.MultipleLocator(0.1)  # Every 0.1 degrees

    # Add MID LEVEL gridlines (half degrees) - standard width with labels
    gl_mid = ax.gridlines(draw_labels=True, alpha=0.8, zorder=12, linewidth=0.3)
    gl_mid.xlocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees
    gl_mid.ylocator = mticker.MultipleLocator(0.5)  # Every 0.5 degrees

    # Add MAJOR gridlines (full degrees) - thick with labels
    gl_major = ax.gridlines(draw_labels=True, alpha=0.8, zorder=13, linewidth=0.6)
    gl_major.xlocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    gl_major.ylocator = mticker.MultipleLocator(1.0)  # Every 1.0 degrees
    
    # Format labels
    gl_mid.xformatter = LONGITUDE_FORMATTER
    gl_mid.yformatter = LATITUDE_FORMATTER
    
    gl_major.xformatter = LONGITUDE_FORMATTER
    gl_major.yformatter = LATITUDE_FORMATTER
    
    # Remove labels from top and right
    gl_mid.top_labels = False
    gl_mid.right_labels = False
    gl_mid.bottom_labels = True
    gl_mid.left_labels = True
    
    gl_major.top_labels = False
    gl_major.right_labels = False
    gl_major.bottom_labels = True
    gl_major.left_labels = True

    # calculate how many degrees this range is (different for latitude vs longitude)
    RadiusEarth = 6371 # [km]
    kmPerDegLat = (RadiusEarth * 2 * np.pi) /360
    kmPerDegLon = kmPerDegLat * np.cos( np.deg2rad(RadarXR.latitude))
    
    BoxRangeDegLat = BoxRange / kmPerDegLat
    BoxRangeDegLon = BoxRange / kmPerDegLon

    plt.xlim([float(RadarXR.longitude)  - BoxRangeDegLon, float(RadarXR.longitude)  + BoxRangeDegLon])
    plt.ylim([float(RadarXR.latitude)   - BoxRangeDegLat,  float(RadarXR.latitude)  + BoxRangeDegLat])
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    plt.title(VarName + ' for ' + RadarSiteName + ' Radar\nfor the ' + str(ElevationAngle) + '° Elevation Angle\non ' + \
              RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
              str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')

    SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + PlotType + '/' + RadarIDno + '/' + RadarFileDate + '/'
    SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + VarNameLong + '_' + \
                 PlotType + str(ElevationAngle) + 'deg.png'
    
    SavePath = SaveFolder + SaveFile
    
    if not Path(SaveFolder).exists():
        print('doing')
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    # plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
    # plt.close()

In [ ]:
plt.hist(PlotVar.values[~np.isnan(PlotVar.values)],bins=200)


In [ ]:
RadarXR